<03_Rotten_Apple.ipynb>

제미나이 의존도: 40-50%

제가 해보고자하는 주제는 신선한 사과 분류기이기 때문에, 이번엔 상한 사과를 크롤링하는 코드입니다.

01_Crawling.ipynb에서 신선한 사과를 크롤링할 때, 리캡차가 뜨는 걸 보고, IP 차단이 두려워서 구글링 하다가 user-agent argument 설정에 대해 알게 되고, 이 방법을 시도했습니다.
user-agent는 크롤링 봇이 서버에 접속할 때 기기의 정보를 담아 보내는 '자기소개' 입니다.

차단을 방지할 수 있다는 장점이 있지만, 이번에 제가 써봄으로써 여전히 리캡차는 떴습니다. 그래도 차단당하지 않았다는 것에 안도감을 느낍니다.

신선한 사과때와는 다르게 BeautifulSoup을 써서 크롤링해 봤습니다.
BeautifulSoup은 HTML 문서에서 데이터를 빠르고 쉽게 추출할 수 있도록 도와주는 웹 크롤링 라이브러리 입니다.

In [1]:
# 계획..
# 상한 사과 이미지를 가져온다. 크롤링을 통해서.
import selenium
print(selenium.__version__)

4.44.0


In [11]:
import time
import random
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options

options = Options()

user_agent = "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36"
options.add_argument('user-agent=' + user_agent)

driver = webdriver.Chrome(options=options)
driver.get("https://google.com")

time.sleep(3)

In [12]:
from selenium.webdriver.common.keys import Keys

search_box = driver.find_element(By.NAME, "q")
search_box.send_keys("상한 사과")
time.sleep(random.uniform(1.0, 2.5))
search_box.send_keys(Keys.ENTER)

In [14]:
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

image_tab = WebDriverWait(driver, 10).until(
    EC.element_to_be_clickable((By.LINK_TEXT, "이미지"))
)
image_tab.click()

time.sleep(2.0)

In [21]:
import requests
from bs4 import BeautifulSoup

# 셀레니움이 열어둔 현재 페이지의 HTML 소스코드 통째로 가져오기
html = driver.page_source

# 가져온 HTML을 BeautifulSoup에게 먹여서 분석용 객체로 만들기
soup = BeautifulSoup(html, 'html.parser')

target_images = soup.find_all('img', class_=False)
print(len(target_images))

print(type(target_images))


100
<class 'bs4.element.ResultSet'>


In [22]:
for i, image in enumerate(target_images):
    img_src = image.get('src')
    print(f"src: {img_src}")
    if i >= 9:
        break

src: /logos/doodles/2026/world-cup-2026-the-art-of-the-bicycle-kick-617-6753651837111107-sdrk.png
src: https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcSGBU6JhzvgXt9LgOIclql51sEE6qEyQHfXI7291iRicA&s=10
src: https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcQMQNE-JoKPHTWHNiI8ldkc5tBscV0wHInkSsOUU18wFw&s=10
src: https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcR0rFK3fxpbZ47qOQSGBefcLRiqh0gKRtdhIdX8xhWqJw&s=10
src: https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcT8Tk1Zi-jgfrJdcTBSaiCaeoBfc5mUxRRzJW7wEDDGTA&s=10
src: https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRa4LSdCz7gZ8iR2imY93CKQoRRRg8I9n9ZugmMOgT25A&s=10
src: https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcSbGPeqyqEuUP1wkWEqEHnRIbuU_xh1sPZy37zTpc5WeQ&s=10
src: https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcQm6HgdAYXuzgZ9VX7zFWKkL-J_ghcJSzaftrKhMNaTkQ&s
src: https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcS3pqkMsDgzv5JOG7Fjeg4lzoE_ZJsnu1LJqo1-fILc3A&s=10
src: https://encrypted-tbn0.gstat

In [23]:
image_url_list = []
for image in target_images:
    img_src = image.get('src')
    image_url_list.append(img_src)
print(len(image_url_list))

100


In [1]:
!pwd

/Users/jeongjaehun/Crawling


In [30]:
import urllib.request
not_type_count = 0
for i, url in enumerate(image_url_list):
    if url.startswith("https"):
        urllib.request.urlretrieve(url, f"/Users/jeongjaehun/Crawling/rotten_apple/rtn_apple_{i}.jpg")
        time.sleep(random.uniform(0.5, 1.5))
    else:
        not_type_count += 1
        continue
print(not_type_count)

72


In [9]:
import requests

headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36"
}
url = "https://www.google.com/search?q=rotten+apple&tbm=isch"

response = requests.get(url, headers=headers)
print(response.status_code)

200
